# Possession models, final training: 5 architectures x 3 seeds, then the Metrica sample-game test
Rebuilds everything from public data (nothing local, nothing from the competition match):
IDSSE (Bundesliga, CC BY 4.0) + SoccerTrack v2 (CC BY 4.0) -> trains GRU, LSTM, TCN, ConvGRU and Transformer on ALL of it (3 seeds each) -> scores them on Metrica's public sample games (independent data, never trained on).

**Settings before running:** Internet ON, **GPU on** (T4 is fine; without it training is much slower), Secret `HF_TOKEN` (Hugging Face read token; accept the terms at huggingface.co/datasets/atomscott/soccertrack-v2 first).
Models are saved to `/kaggle/working/out` as soon as each one finishes, so download them from the file panel as they appear. Run the cells top to bottom.

In [ ]:
# ---- settings you may edit ----
EPOCHS = {"gru": 25, "lstm": 28, "tcn": 40, "convgru": 25, "transformer": 25}   # epochs per architecture
SEEDS = 3

In [ ]:
import os, sys, glob, subprocess, re
TEMP = "/kaggle/temp" if os.path.isdir("/kaggle/temp") else "/tmp/work"
ROOT = TEMP + "/repo"
OUTW = "/kaggle/working/out" if os.path.isdir("/kaggle/working") else TEMP + "/out"
for d in ("training", "evaluation", "data/external/idsse", "data/external/soccertrack"):
    os.makedirs(os.path.join(ROOT, d), exist_ok=True)
os.makedirs(OUTW, exist_ok=True)
os.environ["PYTHONWARNINGS"] = "ignore"
def sh(cmd):
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True); return (r.stdout + r.stderr)[-600:]
print(sh("pip -q install kloppy ijson huggingface_hub"))
metrics = ROOT + "/evaluation/possession_metrics.py"
if not os.path.exists(metrics):   # the official scoring code from the public challenge repo
    print(sh("curl -sSL -o %s https://raw.githubusercontent.com/metrica-sports/more-than-a-hack-2027/main/evaluation/possession_metrics.py" % metrics))
print("metrics file:", os.path.getsize(metrics), "bytes | working dir:", ROOT)
import torch; print("GPU available:", torch.cuda.is_available())

## 1. Source files

In [ ]:
def write(rel, text):
    path = os.path.join(ROOT, rel); os.makedirs(os.path.dirname(path), exist_ok=True)
    open(path, "w", encoding="utf-8").write(text); print("wrote", rel, len(text), "chars")

In [ ]:
write('training/build_idsse.py', r'''"""Download the open IDSSE / Sportec Bundesliga matches and turn each into a training file.

EXTERNAL data only (7 other matches, CC-BY 4.0, DFL / Sportec Solutions). Nothing from the
Barcelona-Athletic match is touched here, so it is fine to run this anywhere, Colab included.

For every frame we keep:
    ball    (n, 2)      ball x,y, normalised 0-1 (same convention as the competition ATD)
    players (n, K, 2)   up to K tracked players x,y, NaN where unused
    team    (n, K)      0 = home, 1 = away, -1 = unused slot
    label   (n,)        0 = H, 1 = A, 2 = D   <- from the tracking's own ball-state / owner fields
    period  (n,)        1 or 2

The label comes from fields that Sportec's tracking already carries (ball alive/dead and which
team owns the ball) -- the same kind of source the competition's ground truth is derived from.

    pip install kloppy numpy
    python training/build_idsse.py                  # all 7 matches, ~400 MB of XML each
    python training/build_idsse.py --matches J03WPY # just one, to test first

Each match takes a few minutes and a few GB of RAM to parse.
"""
import argparse
import os

import numpy as np
from kloppy import sportec
from kloppy.domain import BallState, Team

MATCHES = ["J03WMX", "J03WN1", "J03WOH", "J03WOY", "J03WPY", "J03WQQ", "J03WR9"]
K = 26                      # player slots per frame (22 + a few extras for subs/officials)
HERE = os.path.dirname(os.path.abspath(__file__))
OUT_DIR = os.path.join(os.path.dirname(HERE), "data", "external", "idsse")   # data/ is gitignored


def convert(match_id):
    print("loading", match_id, "(downloads and parses a large XML, be patient) ...")
    ds = sportec.load_open_tracking_data(match_id=match_id, coordinates="metrica")
    n = len(ds.frames)
    ball = np.full((n, 2), np.nan, dtype=np.float32)
    players = np.full((n, K, 2), np.nan, dtype=np.float32)
    team = np.full((n, K), -1, dtype=np.int8)
    label = np.full(n, 2, dtype=np.int8)
    period = np.zeros(n, dtype=np.int8)

    for i, frame in enumerate(ds.frames):
        period[i] = frame.period.id
        if frame.ball_coordinates is not None:
            ball[i] = (frame.ball_coordinates.x, frame.ball_coordinates.y)
        owner = frame.ball_owning_team
        if frame.ball_state == BallState.ALIVE and owner is not None:
            label[i] = 0 if owner.ground.value == "home" else 1
        # else stays 2 = dead ball
        slot = 0
        for player, data in frame.players_data.items():
            if data.coordinates is None or slot >= K:
                continue
            players[i, slot] = (data.coordinates.x, data.coordinates.y)
            team[i, slot] = 0 if player.team.ground.value == "home" else 1
            slot += 1

    counts = {name: int(np.sum(label == code)) for name, code in (("H", 0), ("A", 1), ("D", 2))}
    print("  frames {}  labels {}  ball visible {:.1f}%".format(
        n, counts, 100.0 * np.mean(~np.isnan(ball[:, 0]))))
    return dict(ball=ball, players=players, team=team, label=label, period=period)


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--matches", nargs="+", default=MATCHES)
    args = parser.parse_args()
    os.makedirs(OUT_DIR, exist_ok=True)
    for match_id in args.matches:
        path = os.path.join(OUT_DIR, match_id + ".npz")
        if os.path.exists(path):
            print("skip", match_id, "(already built)")
            continue
        np.savez_compressed(path, **convert(match_id))
        print("  wrote", path)


if __name__ == "__main__":
    main()
''')

In [ ]:
write('training/train_possession.py', r'''"""Train a small sequence model that labels every moment H / A / D from tracking.

Trains ONLY on the external IDSSE matches built by build_idsse.py. Nothing from the competition
match is used here. One IDSSE match is held out to check that the model generalises.

Because IDSSE has the ball on 100% of frames and the competition tracking loses it on ~41%, every
training copy gets random ball dropouts applied so the model learns to cope with gaps.

    pip install torch numpy
    python training/train_possession.py                  # holds out J03WPY, 6 matches train
    python training/train_possession.py --epochs 30 --holdout J03WQQ

Runs on CPU in minutes; a GPU (Colab) just makes it faster. Saves training/possession_gru_v2.pt.
"""
import argparse
import glob
import os

import numpy as np
import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))
DATA = os.path.join(os.path.dirname(HERE), "data", "external", "idsse")
PITCH = np.array([105.0, 68.0], dtype=np.float32)
STEP = 5                 # model runs at 5 fps; labels are expanded back to 25 fps afterwards
WIN, STRIDE = 300, 150   # 300 steps = 60 s windows
N_FEATURES = 12


def dropout_mask(n, rng, mean_gap=None):
    """True where the ball is hidden: a MIX of short flickers and long outages (4 s to 60 s).

    The target share of hidden frames is drawn per copy (30-50%) so the model sees a range.
    These are generic ranges, not fitted to the competition tracking.
    """
    target = rng.uniform(0.30, 0.50)
    hidden = np.zeros(n, dtype=bool)
    while hidden.mean() < target:
        start = rng.integers(0, n)
        if rng.random() < 0.4:
            length = int(rng.exponential(25)) + 1                       # short flicker
        else:
            length = int(np.exp(rng.uniform(np.log(100), np.log(1500))))  # 4 s .. 60 s outage
        hidden[start:start + length] = True
    return hidden


def features(ball, players, team, hidden):
    """(n, N_FEATURES) per-frame features that do not depend on which way a team attacks."""
    n = len(ball)
    seen = ~hidden & ~np.isnan(ball[:, 0])
    # carry the last seen ball forward through gaps; remember how stale it is
    idx = np.where(seen, np.arange(n), 0)
    np.maximum.accumulate(idx, out=idx)
    b = ball[idx] * PITCH
    age = np.minimum(np.arange(n) - idx, 250) / 250.0
    age[~np.maximum.accumulate(seen)] = 1.0          # nothing seen yet

    d = np.linalg.norm(players * PITCH - b[:, None, :], axis=2)       # (n, K)
    d = np.where(np.isnan(d), np.inf, d)
    home = np.where(team == 0, d, np.inf)
    away = np.where(team == 1, d, np.inf)
    dh = np.minimum(home.min(axis=1), 40.0)
    da = np.minimum(away.min(axis=1), 40.0)
    near5h, near5a = (home < 5).sum(1), (away < 5).sum(1)
    near12h, near12a = (home < 12).sum(1), (away < 12).sum(1)

    step = 5
    speed = np.zeros(n, dtype=np.float32)
    speed[step:] = np.linalg.norm(b[step:] - b[:-step], axis=1) / (step / 25.0)
    pm = players * PITCH
    pspeed = np.full(players.shape[:2], np.nan, dtype=np.float32)
    pspeed[step:] = np.linalg.norm(pm[step:] - pm[:-step], axis=2) / (step / 25.0)
    pspeed = np.where(pspeed > 12.0, np.nan, pspeed)       # ignore identity swaps / teleports
    with np.errstate(all="ignore"):
        mean_pspeed = np.nan_to_num(np.nanmean(pspeed, axis=1), nan=0.0)
        frac_still = np.nan_to_num(np.nanmean(pspeed < 1.0, axis=1), nan=0.0)
    out = np.stack([
        seen.astype(np.float32), age, dh / 10.0, da / 10.0,
        (da - dh) / 10.0,                       # >0 means home is closer
        near5h / 3.0, near5a / 3.0, near12h / 6.0, near12a / 6.0,
        np.minimum(speed, 40.0) / 20.0,
        mean_pspeed / 3.0, frac_still,
    ], axis=1)
    # A ball that is untracked from the very first frame of a half leaves NaN in the speed
    # feature; one NaN poisons a whole recurrent window. Missing means 0.
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def load(path, rng):
    z = np.load(path)
    sigma = rng.uniform(0.1, 0.5) / PITCH                  # metres of random tracking jitter
    ball = z["ball"] + rng.normal(0, 1, z["ball"].shape).astype(np.float32) * sigma
    players = z["players"] + rng.normal(0, 1, z["players"].shape).astype(np.float32) * sigma
    x = features(ball, players, z["team"], dropout_mask(len(z["ball"]), rng))
    return x[::STEP], z["label"][::STEP].astype(np.int64)


def windows(x, y):
    out = []
    for s in range(0, max(1, len(x) - WIN + 1), STRIDE):
        out.append((x[s:s + WIN], y[s:s + WIN]))
    return out


class GRU(nn.Module):
    def __init__(self, hidden=64):
        super().__init__()
        self.rnn = nn.GRU(N_FEATURES, hidden, num_layers=2, batch_first=True,
                          bidirectional=True, dropout=0.1)
        self.head = nn.Linear(2 * hidden, 3)

    def forward(self, x):
        return self.head(self.rnn(x)[0])


def macro_f1(y, p):
    f = []
    for c in range(3):
        tp = np.sum((y == c) & (p == c)); fp = np.sum((y != c) & (p == c)); fn = np.sum((y == c) & (p != c))
        f.append(0.0 if tp == 0 else 2 * tp / (2 * tp + fp + fn))
    return float(np.mean(f)), f


def n_possessions(labels):
    """Runs of H or A (dead ball splits them), plus their mean length in seconds at 5 fps."""
    runs, length, prev = [], 0, None
    for v in labels:
        if v in (0, 1):
            if v == prev:
                length += 1
            else:
                if prev in (0, 1): runs.append(length)
                length = 1
        else:
            if prev in (0, 1): runs.append(length)
            length = 0
        prev = v
    if prev in (0, 1): runs.append(length)
    return len(runs), (np.mean(runs) / (25 / STEP) if runs else 0.0)


@torch.no_grad()
def predict(model, x):
    model.eval()
    out = np.zeros(len(x), dtype=np.int64)
    for s in range(0, len(x), WIN):                  # non-overlapping chunks are fine for a check
        chunk = torch.from_numpy(x[s:s + WIN])[None]
        out[s:s + WIN] = model(chunk)[0].argmax(-1).numpy()
    return out


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--holdout", default="J03WPY")
    ap.add_argument("--epochs", type=int, default=20)
    ap.add_argument("--copies", type=int, default=3, help="dropout versions of each match")
    args = ap.parse_args()
    torch.manual_seed(0)
    rng = np.random.default_rng(0)

    paths = sorted(glob.glob(os.path.join(DATA, "*.npz")))
    train_paths = [p for p in paths if args.holdout not in p]
    hold_path = [p for p in paths if args.holdout in p]
    if not train_paths or not hold_path:
        raise SystemExit("need built matches in {} (run build_idsse.py) incl. {}".format(DATA, args.holdout))

    print("building features ...")
    train = []
    for p in train_paths:
        for _ in range(args.copies):
            train += windows(*load(p, rng))
    xh, yh = load(hold_path[0], np.random.default_rng(123))
    print("{} training windows from {} matches; holding out {}".format(len(train), len(train_paths), args.holdout))

    model = GRU()
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    loss_fn = nn.CrossEntropyLoss()
    for epoch in range(args.epochs):
        model.train()
        order = rng.permutation(len(train))
        total = 0.0
        for s in range(0, len(order), 32):
            batch = [train[i] for i in order[s:s + 32]]
            xb = torch.from_numpy(np.stack([b[0] for b in batch]))
            yb = torch.from_numpy(np.stack([b[1] for b in batch]))
            loss = loss_fn(model(xb).reshape(-1, 3), yb.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            total += float(loss) * len(batch)
        pred = predict(model, xh)
        mf, per = macro_f1(yh, pred)
        pn, pd = n_possessions(pred); tn, td = n_possessions(yh)
        print("epoch {:>2}  loss {:.3f}  held-out macroF1 {:.3f} (H {:.2f} A {:.2f} D {:.2f})  "
              "possessions pred {} vs true {}  mean {:.1f}s vs {:.1f}s".format(
                  epoch + 1, total / len(train), mf, per[0], per[1], per[2], pn, tn, pd, td))

    out = os.path.join(HERE, "possession_gru_v2.pt")
    torch.save(model.state_dict(), out)
    print("saved", out)


if __name__ == "__main__":
    main()
''')

In [ ]:
write('training/features2.py', r'''"""v4 features: the 12 ball features of train_possession.py PLUS team-shape features that need
no ball at all, so the model still has something to go on during long ball outages.

Shape features (per team, in the team's own attacking direction): where the block sits (centroid,
front line, back line), how wide / deep it is, how fast it moves, how many players sprint, and
how fast it moves forward; plus a few joint ones. They are smoothed over 1 s.

Attacking direction is found from the tracking itself, per period (team 0 is the one whose mean
x is lower, so it attacks +x). No match-specific constants, no labels.
"""
import warnings

import numpy as np

from train_possession import features as ball_features, PITCH, STEP

N_SHAPE = 23
N_FEATURES2 = 12 + N_SHAPE
SMOOTH = 5                     # steps (= 1 s at 5 fps)


def orient(players, team, period):
    """Copy of players (normalised 0-1) flipped per period so team 0 sits on the left."""
    out = players.copy()
    period = np.zeros(len(players), dtype=np.int8) if period is None else period
    for p in np.unique(period):
        m = period == p
        tm = team[m]
        x = players[m][..., 0]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            x0 = np.nanmean(np.where(tm == 0, x, np.nan))
            x1 = np.nanmean(np.where(tm == 1, x, np.nan))
        if np.isfinite(x0) and np.isfinite(x1) and x0 > x1:
            out[m] = 1.0 - out[m]
    return out


def _smooth(a):
    """Moving average over SMOOTH steps along axis 0, ignoring NaN."""
    ok = np.isfinite(a).astype(np.float32)
    a0 = np.where(np.isfinite(a), a, 0.0)
    k = np.ones(SMOOTH, dtype=np.float32)
    num = np.stack([np.convolve(a0[:, j], k, "same") for j in range(a.shape[1])], 1)
    den = np.stack([np.convolve(ok[:, j], k, "same") for j in range(a.shape[1])], 1)
    return np.where(den > 0, num / np.maximum(den, 1e-6), 0.0)


def shape_features(players, team, period):
    """(ceil(n/STEP), N_SHAPE) ball-free team-shape features at STEP resolution."""
    n = len(players)
    po = orient(players, team, period) * PITCH                   # metres, team 0 attacks +x
    idx = np.arange(0, n, STEP)
    prev = np.maximum(idx - STEP, 0)
    dt = STEP / 25.0
    cur, old = po[idx], po[prev]
    tm = team[idx]
    delta = cur - old
    spd = np.linalg.norm(delta, axis=2) / dt
    spd = np.where((spd > 12.0) | (idx[:, None] < STEP), np.nan, spd)
    cols = []
    sx_sy = []
    cx_abs = []
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        for t in (0, 1):
            m = (tm == t) & np.isfinite(cur[..., 0])
            sign = 1.0 if t == 0 else -1.0
            fx = np.where(m, cur[..., 0] if t == 0 else 105.0 - cur[..., 0], np.nan)
            y = np.where(m, cur[..., 1], np.nan)
            s = np.where(m, spd, np.nan)
            fv = np.where(m, sign * delta[..., 0] / dt, np.nan)
            fv = np.where(np.isfinite(s), fv, np.nan)
            sx, sy = np.nanstd(fx, axis=1) / 15.0, np.nanstd(y, axis=1) / 15.0
            cols += [(np.nanmean(fx, 1) - 52.5) / 30.0, sx, sy,
                     (np.nanpercentile(fx, 80, axis=1) - 52.5) / 30.0,
                     (np.nanpercentile(fx, 20, axis=1) - 52.5) / 30.0,
                     np.nanmean(s, 1) / 3.0, np.nanmean(s > 4.0, 1) if False else
                     np.nansum(np.where(np.isfinite(s), s > 4.0, 0), 1) / np.maximum(np.isfinite(s).sum(1), 1),
                     np.nanmean(fv, 1) / 2.0, m.sum(1) / 11.0]
            sx_sy.append(sx * sy)
            cx_abs.append(np.nanmean(np.where(m, cur[..., 0], np.nan), 1))
        alive = np.isfinite(spd)
        joint = [(cx_abs[0] - cx_abs[1]) / 30.0, sx_sy[0] - sx_sy[1],
                 np.nanmean(spd, 1) / 3.0, np.nanpercentile(spd, 90, axis=1) / 6.0,
                 np.nansum(np.where(alive, spd > 4.0, 0), 1) / np.maximum(alive.sum(1), 1)]
    out = np.stack(cols + joint, axis=1).astype(np.float32)
    out = _smooth(out)
    return np.nan_to_num(out, nan=0.0, posinf=0.0, neginf=0.0).astype(np.float32)


def features2(ball, players, team, hidden, period=None):
    """(ceil(n/STEP), N_FEATURES2): ball features (subsampled) + shape features."""
    a = ball_features(ball, players, team, hidden)[::STEP]
    b = shape_features(players, team, period)
    assert len(a) == len(b), (len(a), len(b))
    return np.concatenate([a, b], axis=1)


def make_gru2(hidden=64, n_in=N_FEATURES2):
    import torch.nn as nn

    class GRU2(nn.Module):
        def __init__(self):
            super().__init__()
            self.rnn = nn.GRU(n_in, hidden, num_layers=2, batch_first=True,
                              bidirectional=True, dropout=0.1)
            self.head = nn.Linear(2 * hidden, 3)

        def forward(self, x):
            return self.head(self.rnn(x)[0])

    return GRU2()


def make_model(arch="gru", n_in=N_FEATURES2, hidden=64):
    """Sequence models with the same interface: (batch, time, n_in) -> (batch, time, 3) logits."""
    import torch, torch.nn as nn
    if arch == "gru":
        return make_gru2(hidden, n_in)

    class LSTM(nn.Module):
        def __init__(self):
            super().__init__()
            self.rnn = nn.LSTM(n_in, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
            self.head = nn.Linear(2 * hidden, 3)
        def forward(self, x):
            return self.head(self.rnn(x)[0])

    class Block(nn.Module):
        def __init__(self, c, dil):
            super().__init__()
            self.conv = nn.Conv1d(c, c, 5, padding=2 * dil, dilation=dil)
            self.norm = nn.GroupNorm(1, c)
            self.drop = nn.Dropout(0.1)
        def forward(self, x):
            return x + self.drop(torch.relu(self.norm(self.conv(x))))

    class TCN(nn.Module):          # dilated non-causal convolutions, receptive field about 125 steps (25 s)
        def __init__(self):
            super().__init__()
            self.inp = nn.Conv1d(n_in, hidden, 1)
            self.blocks = nn.Sequential(*[Block(hidden, d) for d in (1, 2, 4, 8, 16)])
            self.head = nn.Conv1d(hidden, 3, 1)
        def forward(self, x):
            return self.head(self.blocks(self.inp(x.transpose(1, 2)))).transpose(1, 2)

    class ConvGRU(nn.Module):      # local conv smoothing in front of a bidirectional GRU
        def __init__(self):
            super().__init__()
            self.conv = nn.Sequential(nn.Conv1d(n_in, hidden, 5, padding=2), nn.ReLU())
            self.rnn = nn.GRU(hidden, hidden, num_layers=2, batch_first=True, bidirectional=True, dropout=0.1)
            self.head = nn.Linear(2 * hidden, 3)
        def forward(self, x):
            h = self.conv(x.transpose(1, 2)).transpose(1, 2)
            return self.head(self.rnn(h)[0])

    class Transformer(nn.Module):  # conv embedding + 3 self-attention layers over the 60 s window
        def __init__(self, win=300):
            super().__init__()
            self.emb = nn.Sequential(nn.Conv1d(n_in, hidden, 5, padding=2), nn.ReLU())
            self.pos = nn.Parameter(torch.zeros(1, win, hidden))
            layer = nn.TransformerEncoderLayer(hidden, 4, 4 * hidden, 0.1, batch_first=True)
            self.enc = nn.TransformerEncoder(layer, 3)
            self.head = nn.Linear(hidden, 3)
        def forward(self, x):
            h = self.emb(x.transpose(1, 2)).transpose(1, 2)
            return self.head(self.enc(h + self.pos[:, :h.shape[1]]))

    return {"lstm": LSTM, "tcn": TCN, "convgru": ConvGRU, "transformer": Transformer}[arch]()
''')

In [ ]:
write('training/decode.py', r'''"""Turn the model's per-step probabilities into a clean H/A/D sequence.

A Viterbi pass with switch penalties: changing team (H<->A) costs `lam`, entering or leaving
dead ball costs `mu`. Bigger penalties give fewer, longer possessions. The penalties are chosen on
EXTERNAL held-out matches only (see train_v3.py), never on the competition match.
"""
import numpy as np
import torch

from train_possession import WIN


@torch.no_grad()
def half_logprobs(model, x):
    """(n, 3) log-probabilities from overlapping windows, averaged so edges don't decide a step."""
    model.eval()
    acc = np.zeros((len(x), 3), dtype=np.float32)
    cnt = np.zeros(len(x), dtype=np.float32)
    starts = list(range(0, max(1, len(x) - WIN + 1), WIN // 2))
    if starts[-1] + WIN < len(x):
        starts.append(len(x) - WIN)
    for s in starts:
        out = torch.log_softmax(model(torch.from_numpy(x[s:s + WIN])[None])[0], -1).numpy()
        acc[s:s + WIN] += out
        cnt[s:s + WIN] += 1
    out = acc / np.maximum(cnt, 1)[:, None]
    if np.isnan(out).any():
        raise ValueError("NaN in model output: {} values. Check the input features.".format(int(np.isnan(out).sum())))
    return out


def viterbi(logp, lam, mu):
    """Best state path (0=H, 1=A, 2=D) under switch penalties lam (H<->A) and mu (to/from D)."""
    n = len(logp)
    cost = np.array([[0, lam, mu],
                     [lam, 0, mu],
                     [mu, mu, 0]], dtype=np.float32)
    score = logp[0].copy()
    back = np.zeros((n, 3), dtype=np.int8)
    for t in range(1, n):
        cand = score[:, None] - cost                     # cand[i, j]: come from i, go to j
        back[t] = cand.argmax(0)
        score = cand.max(0) + logp[t]
    path = np.zeros(n, dtype=np.int8)
    path[-1] = score.argmax()
    for t in range(n - 1, 0, -1):
        path[t - 1] = back[t, path[t]]
    return path
''')

In [ ]:
write('training/train_v4.py', r'''"""Train v4: GRU on ball features + ball-free team-shape features, on EXTERNAL data only.

    IDSSE (7 Bundesliga matches, ball tracked -> we simulate outages)
    SoccerTrack v2 (10 matches, ball always hidden, labels may be -1 = unknown -> ignored)

Phase 1 (default): hold out one IDSSE match and one SoccerTrack match, pick the epoch by the
official spell F1 after Viterbi decoding (fixed lam/mu), report on a second IDSSE test match.
Phase 2 (--final): train on everything for --epochs epochs with several seeds, saving
training/possession_gru_v4_s{seed}.pt.

    python training/train_v4.py                       # phase 1
    python training/train_v4.py --final --epochs 25   # phase 2 (use the best epoch from phase 1)
"""
import argparse
import glob
import os
import sys

import numpy as np
import torch
import torch.nn as nn

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(ROOT, "evaluation"))
import possession_metrics as pm  # noqa: E402
from train_possession import features as ball_feats, DATA as IDSSE, PITCH, STEP, WIN, STRIDE, dropout_mask  # noqa: E402
from features2 import features2, make_model  # noqa: E402
from decode import half_logprobs, viterbi  # noqa: E402

ST_DIR = os.path.join(ROOT, "data", "external", "soccertrack")
LAM, MU = 2.0, 4.0           # decoding penalties (chosen on IDSSE in v3)
ARCH = "gru"
N_IN = 35                    # 35 = ball + team-shape features, 12 = ball features only (old model)


def load2(path, rng, soccertrack):
    z = np.load(path)
    sigma = rng.uniform(0.1, 0.5) / PITCH
    ball = z["ball"] + rng.normal(0, 1, z["ball"].shape).astype(np.float32) * sigma
    players = z["players"] + rng.normal(0, 1, z["players"].shape).astype(np.float32) * sigma
    team = z["team"]
    if team.ndim == 1:
        team = np.broadcast_to(team, z["players"].shape[:2])
    n = len(ball)
    hidden = np.ones(n, bool) if soccertrack else dropout_mask(n, rng)
    if N_IN <= 12:
        x = ball_feats(ball, players, team, hidden)[::STEP]
    else:
        x = features2(ball, players, team, hidden, z["period"])[:, :N_IN]
    return x, z["label"][::STEP].astype(np.int64)


def windows(x, y):
    out = []
    for s in range(0, max(1, len(x) - WIN + 1), STRIDE):
        xs, ys = x[s:s + WIN], y[s:s + WIN]
        if (ys >= 0).mean() > 0.3:
            out.append((xs, ys))
    return out


def spell(y, logp):
    path = viterbi(logp, LAM, MU).astype(np.int8)
    gt = np.where(y < 0, pm.X, y).astype(np.int8)
    runs = pm.valid_runs(gt)
    if len(runs) == 0:
        return float("nan")
    return pm.spell_scores(gt, path, runs)["mean_f1"]


def train_one(train, epochs, seed, evaluate=None, save_path=None):
    torch.manual_seed(seed)
    rng = np.random.default_rng(seed)
    model = make_model(ARCH, n_in=N_IN)
    dev = torch.device("cuda" if (torch.cuda.is_available() and evaluate is None) else "cpu")
    model.to(dev)
    opt = torch.optim.Adam(model.parameters(), lr=2e-3)
    loss_fn = nn.CrossEntropyLoss(ignore_index=-1)
    best, best_ep = -1.0, 0
    for ep in range(1, epochs + 1):
        model.train()
        order = rng.permutation(len(train))
        total = 0.0
        for s in range(0, len(order), 32):
            batch = [train[i] for i in order[s:s + 32]]
            xb = torch.from_numpy(np.stack([b[0] for b in batch])).to(dev)
            yb = torch.from_numpy(np.stack([b[1] for b in batch])).to(dev)
            loss = loss_fn(model(xb).reshape(-1, 3), yb.reshape(-1))
            opt.zero_grad(); loss.backward(); opt.step()
            total += float(loss.detach()) * len(batch)
        msg = "epoch {:>2}  loss {:.3f}".format(ep, total / len(train))
        if evaluate:
            score, detail = evaluate(model)
            msg += "  val " + detail
            if score > best:
                best, best_ep = score, ep
                torch.save(model.state_dict(), save_path)
                msg += "  <- best, saved"
        print(msg, flush=True)
    model.cpu()
    return model, best, best_ep


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--val-idsse", default="J03WPY")
    ap.add_argument("--test-idsse", default="J03WQQ")
    ap.add_argument("--val-st", default="132877")
    ap.add_argument("--epochs", type=int, default=30)
    ap.add_argument("--copies", type=int, default=2)
    ap.add_argument("--final", action="store_true")
    ap.add_argument("--seeds", type=int, default=3)
    ap.add_argument("--n-in", type=int, default=35)
    ap.add_argument("--no-st", action="store_true", help="train on IDSSE only")
    ap.add_argument("--st-copies", type=int, default=1)
    ap.add_argument("--tag", default="v4")
    ap.add_argument("--archs", default=None, help="final mode only: arch:epochs,arch:epochs,... (features built once)")
    ap.add_argument("--arch", default="gru", choices=["gru", "lstm", "tcn", "convgru", "transformer"])
    ap.add_argument("--out-dir", default=HERE)
    args = ap.parse_args()
    global N_IN, ARCH
    N_IN = args.n_in
    ARCH = args.arch
    os.makedirs(args.out_dir, exist_ok=True)

    idsse = sorted(glob.glob(os.path.join(IDSSE, "*.npz")))
    st = sorted(glob.glob(os.path.join(ST_DIR, "*.npz")))
    if not idsse or not st:
        raise SystemExit("need IDSSE npz in {} and SoccerTrack npz in {}".format(IDSSE, ST_DIR))
    held = [] if args.final else [args.val_idsse, args.test_idsse, args.val_st]
    rng = np.random.default_rng(0)

    print("building features (a few minutes) ...", flush=True)
    train = []
    for p in idsse:
        if any(h in p for h in held):
            continue
        for _ in range(args.copies):
            train += windows(*load2(p, rng, False))
    for p in st:
        if any(h in p for h in held):
            continue
        for _ in range(0 if args.no_st else args.st_copies):
            train += windows(*load2(p, rng, True))
    print("{} training windows".format(len(train)), flush=True)

    if args.final and args.archs:
        for item in args.archs.split(","):
            arch, ep = item.split(":")
            ARCH = arch
            for seed in range(args.seeds):
                out = os.path.join(args.out_dir, "possession_gru_{}_final_s{}.pt".format(arch, seed))
                print("\n=== {} seed {} ({} epochs) ===".format(arch, seed, ep), flush=True)
                model, _, _ = train_one(train, int(ep), seed)
                torch.save(model.state_dict(), out)
                print("saved", out, flush=True)
        return
    if args.final:
        for seed in range(args.seeds):
            out = os.path.join(args.out_dir, "possession_gru_{}_s{}.pt".format(args.tag, seed))
            model, _, _ = train_one(train, args.epochs, seed)
            torch.save(model.state_dict(), out)
            print("saved", out, flush=True)
        return

    vi = load2([p for p in idsse if args.val_idsse in p][0], np.random.default_rng(123), False)
    ti = load2([p for p in idsse if args.test_idsse in p][0], np.random.default_rng(456), False)
    vs = [load2(p, np.random.default_rng(7), True) for p in st if args.val_st in p]
    if not vs:
        raise SystemExit("no SoccerTrack match containing " + args.val_st)

    def score_all(model, with_test=False):
        a = spell(vi[1], half_logprobs(model, vi[0]))
        b = float(np.nanmean([spell(y, half_logprobs(model, x)) for x, y in vs]))
        return a, b

    def evaluate(model):
        a, b = score_all(model)
        return (a + b) / 2, "IDSSE {:.3f}  SoccerTrack(no ball) {:.3f}  mean {:.3f}".format(a, b, (a + b) / 2)

    path = os.path.join(args.out_dir, "possession_gru_{}_val.pt".format(args.tag))
    model, best, best_ep = train_one(train, args.epochs, 0, evaluate, path)
    model.load_state_dict(torch.load(path, map_location="cpu"))
    print("\nbest epoch {} (val mean spellF1 {:.3f})".format(best_ep, best))
    print("TEST IDSSE {} (never used to choose anything): spellF1 {:.3f}".format(
        args.test_idsse, spell(ti[1], half_logprobs(model, ti[0]))))
    print("RESULT arch={} tag={} n_in={} no_st={} best_epoch={} val={:.3f} test_idsse={:.3f}".format(
        args.arch, args.tag, args.n_in, args.no_st, best_ep, best, spell(ti[1], half_logprobs(model, ti[0]))))


if __name__ == "__main__":
    main()
''')

In [ ]:
write('training/build_metrica.py', r'''"""Metrica Sports public sample games 1 and 2 -> external TEST files (never trained on).

Source: https://github.com/metrica-sports/sample-data  (anonymised; "acknowledge the source").
Ball is tracked on essentially every frame; labels are derived from the hand-collected event data:

  * a team owns the ball from the start of its PASS / SHOT / RECOVERY / SET PIECE until the next
    event, if that gap is at most 10 s; longer gaps stay unknown (-1) and are not scored
  * after a BALL OUT, or after a goal, the ball is dead until the next event (at most 90 s)
  * the gap before a free kick is left unknown (the whistle is not an event)

    python training/build_metrica.py        # downloads about 130 MB, writes data/external/metrica/
"""
import os
import urllib.request

import numpy as np
import pandas as pd

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
RAW = os.path.join(ROOT, "data", "external", "metrica_raw")
OUT = os.path.join(ROOT, "data", "external", "metrica")
URL = "https://raw.githubusercontent.com/metrica-sports/sample-data/master/data/Sample_Game_{g}/Sample_Game_{g}_{f}.csv"
FPS = 25
OWN = {"PASS", "SHOT", "RECOVERY", "SET PIECE"}
MAX_OWN_GAP, MAX_DEAD = 10 * FPS, 90 * FPS


def fetch(g, name):
    path = os.path.join(RAW, "g{}_{}.csv".format(g, name))
    if not os.path.exists(path) or os.path.getsize(path) == 0:
        os.makedirs(RAW, exist_ok=True)
        print("downloading", name, "game", g)
        urllib.request.urlretrieve(URL.format(g=g, f=name), path)
    return path


def read_tracking(path):
    d = pd.read_csv(path, skiprows=3, header=None)
    period = d[0].to_numpy(np.int8)
    xy = d.iloc[:, 3:].to_numpy(np.float32)          # player x,y pairs..., then ball x,y
    return period, xy[:, :-2].reshape(len(d), -1, 2), xy[:, -2:]


def build_labels(ev, n):
    """0 home, 1 away, 2 dead, -1 unknown, at tracking frame resolution."""
    ev = ev.sort_values(["Start Frame", "End Frame"], kind="stable").reset_index(drop=True)
    lab = np.full(n, -1, dtype=np.int8)
    rows = list(ev.itertuples(index=False))
    use = [r for r in rows if r.Type in OWN or r.Type == "BALL OUT"]
    for k in range(len(use) - 1):
        r, nxt = use[k], use[k + 1]
        f, f2 = max(int(r._4) - 1, 0), min(int(nxt._4) - 1, n)
        if f2 <= f:
            continue
        team = 0 if r.Team == "Home" else 1
        goal = r.Type == "SHOT" and "GOAL" in str(r.Subtype) and "GOAL KICK" not in str(r.Subtype)
        if r.Type == "BALL OUT" or goal:
            if goal:
                f = max(int(r._6) - 1, f)        # the ball is still live until the shot ends
            if f2 - f <= MAX_DEAD:
                lab[f:f2] = 2
        elif nxt.Type == "SET PIECE" and str(nxt.Subtype) == "FREE KICK":
            continue                              # play stopped for a foul, exact moment unknown
        elif f2 - f <= MAX_OWN_GAP:
            lab[f:f2] = team
    return lab


def convert(g):
    ph, home, ball = read_tracking(fetch(g, "RawTrackingData_Home_Team"))
    _, away, _ = read_tracking(fetch(g, "RawTrackingData_Away_Team"))
    n = len(ph)
    players = np.concatenate([home, away], axis=1)
    team = np.concatenate([np.zeros(home.shape[1]), np.ones(away.shape[1])]).astype(np.int8)
    ev = pd.read_csv(fetch(g, "RawEventsData"))
    lab = build_labels(ev, n)
    kn = lab >= 0
    runs = np.diff(np.r_[0, (lab == 0) | (lab == 1), 0].astype(int))
    print("game {}: frames {} | labelled {:.0%} | home {:.0%} away {:.0%} dead {:.0%} of labelled | ball seen {:.1%}".format(
        g, n, kn.mean(), (lab == 0).sum() / kn.sum(), (lab == 1).sum() / kn.sum(), (lab == 2).sum() / kn.sum(),
        np.mean(~np.isnan(ball[:, 0]))))
    return dict(ball=ball, players=players, team=np.broadcast_to(team, players.shape[:2]).copy(), label=lab, period=ph)


def main():
    os.makedirs(OUT, exist_ok=True)
    for g in (1, 2):
        np.savez_compressed(os.path.join(OUT, "game{}.npz".format(g)), **convert(g))
    print("wrote", OUT)


if __name__ == "__main__":
    main()
''')

In [ ]:
write('training/eval_metrica.py', r'''"""Score trained models on the Metrica sample games (external TEST data, never trained or tuned on).

Each model spec is  name:n_in:arch=glob[,arch=glob...]  -- every weight file matching a glob is loaded
and all of them are averaged (an ensemble; architectures can be mixed).
Two conditions per game: the ball exactly as tracked (about 40% already missing), and with an
extra simulated outage on top (the same generator used in training), averaged over a few seeds.
Scoring uses the official spell F1 (competition metric code) at 5 fps, per half, plus frame macro F1.

    python training/eval_metrica.py \
        v4:gru:35:training/possession_gru_v4_s*.pt \
        old:gru:12:training/possession_gru_final_s*.pt
"""
import argparse
import glob
import os
import sys

import numpy as np
import torch

HERE = os.path.dirname(os.path.abspath(__file__))
ROOT = os.path.dirname(HERE)
sys.path.insert(0, HERE)
sys.path.insert(0, os.path.join(ROOT, "evaluation"))
import possession_metrics as pm  # noqa: E402
from train_possession import features as ball_feats, dropout_mask, macro_f1, STEP, PITCH  # noqa: E402
from features2 import features2, make_model  # noqa: E402
from decode import half_logprobs, viterbi  # noqa: E402

DATA = os.path.join(ROOT, "data", "external", "metrica")
LAM, MU = 2.0, 4.0


def score(y, path):
    gt = np.where(y < 0, pm.X, y).astype(np.int8)
    runs = pm.valid_runs(gt)
    spell = pm.spell_scores(gt, path.astype(np.int8), runs)["mean_f1"] if len(runs) else float("nan")
    m = y >= 0
    return spell, macro_f1(y[m], path[m])[0]


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("models", nargs="+")
    ap.add_argument("--seeds", type=int, default=3)
    args = ap.parse_args()

    specs = []
    for s in args.models:
        name, n_in, parts = s.split(":", 2)
        nets = []
        for part in parts.split(","):
            arch, pattern = part.split("=", 1)
            files = sorted(glob.glob(pattern))
            if not files:
                raise SystemExit("no weights match " + pattern)
            for f in files:
                m = make_model(arch, n_in=int(n_in)); m.load_state_dict(torch.load(f, map_location="cpu")); m.eval()
                nets.append(m)
        specs.append((name, int(n_in), nets))
        print("{}: {} model(s)".format(name, len(nets)))

    games = sorted(glob.glob(os.path.join(DATA, "game*.npz")))
    if not games:
        raise SystemExit("run training/build_metrica.py first")
    results = {(s[0], c): [] for s in specs for c in ("as tracked", "extra outage")}
    for g in games:
        z = np.load(g)
        ball, players, team, label, period = z["ball"], z["players"], z["team"], z["label"], z["period"]
        for cond, seeds in (("as tracked", [0]), ("extra outage", range(args.seeds))):
            for seed in seeds:
                rng = np.random.default_rng(seed)
                hidden = np.isnan(ball[:, 0])
                if cond == "extra outage":
                    hidden = hidden | dropout_mask(len(ball), rng)
                cache = {}
                for name, n_in, nets in specs:
                    key = n_in > 12
                    if key not in cache:
                        cache[key] = features2(ball, players, team, hidden, period) if key else \
                            ball_feats(ball, players, team, hidden)[::STEP]
                    x = cache[key][:, :n_in]
                    y = label[::STEP].astype(np.int64)
                    per = period[::STEP]
                    for p in np.unique(per):
                        sl = per == p
                        if (y[sl] >= 0).mean() < 0.2:
                            continue
                        lp = np.mean([half_logprobs(n, x[sl]) for n in nets], axis=0)
                        results[(name, cond)].append(score(y[sl], viterbi(lp, LAM, MU)))
    print("\n{:<10} {:<14} {:>9} {:>9}   (mean over games x halves x seeds)".format("model", "ball", "spellF1", "macroF1"))
    for (name, cond), v in results.items():
        a = np.array(v, dtype=float)
        print("{:<10} {:<14} {:>9.3f} {:>9.3f}   n={}".format(name, cond, np.nanmean(a[:, 0]), np.nanmean(a[:, 1]), len(a)))


if __name__ == "__main__":
    main()
''')

## 2. IDSSE (7 Bundesliga matches, about 10 min)

In [ ]:
if len(glob.glob(ROOT + "/data/external/idsse/*.npz")) < 7:
    subprocess.run([sys.executable, "-u", ROOT + "/training/build_idsse.py"], check=True)
print(sorted(os.path.basename(p) for p in glob.glob(ROOT + "/data/external/idsse/*.npz")))

## 3. SoccerTrack v2 (20 half-match files from Hugging Face, each deleted after conversion)

In [ ]:
import json, numpy as np, ijson
from huggingface_hub import hf_hub_download
try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    TOKEN = os.environ.get("HF_TOKEN") or input("Hugging Face read token: ")
REPO = "atomscott/soccertrack-v2"
MATCHES = ["117092", "117093", "118575", "118576", "118577", "118578", "128057", "128058", "132831", "132877"]
FPS = 25; K = 26
OWN = {"PASS", "DRIVE", "HIGH PASS", "CROSS", "SHOT", "HEADER", "FREE KICK", "THROW IN", "PLAYER SUCCESSFUL TACKLE"}
DEAD_START = {"OUT", "GOAL"}; MAX_OWN_GAP = 10 * FPS; MAX_DEAD = 90 * FPS

def read_gsr(path):
    with open(path, "rb") as fh: n = int(next(ijson.items(fh, "info"))["seq_length"])
    players = np.full((n, K, 2), np.nan, dtype=np.float32); team = np.full((n, K), -1, dtype=np.int8)
    count = np.zeros(n, dtype=np.int16)
    with open(path, "rb") as fh:
        for ann in ijson.items(fh, "annotations.item", use_float=True):
            if ann.get("supercategory") != "object": continue
            at = ann.get("attributes", {})
            if at.get("role") not in ("player", "goalkeeper"): continue
            f = int(str(ann["image_id"])[-6:]) - 1
            bp = ann.get("bbox_pitch") or {}
            x, y = bp.get("x_bottom_middle"), bp.get("y_bottom_middle")
            if not (0 <= f < n) or x is None or y is None or count[f] >= K: continue
            s = count[f]; count[f] += 1
            players[f, s] = ((x + 52.5) / 105.0, (y + 34.0) / 68.0)
            team[f, s] = 0 if at.get("team") == "left" else 1
    return players, team

def build_labels(actions, n):
    """0 = left team has the ball, 1 = right, 2 = dead ball, -1 = unknown (ignored in training)."""
    ev = sorted((a["frame"] - 1, a["label"], 0 if a["team"] == "left" else 1) for a in actions)
    lab = np.full(n, -1, dtype=np.int8)
    for k in range(len(ev) - 1):
        f, kind, t = ev[k]; f2, nxt = ev[k + 1][0], ev[k + 1][1]
        f, f2 = max(f, 0), min(f2, n)
        if f2 <= f: continue
        if kind in DEAD_START:
            if f2 - f <= MAX_DEAD: lab[f:f2] = 2
        elif kind in OWN and nxt != "FREE KICK" and f2 - f <= MAX_OWN_GAP: lab[f:f2] = t
    return lab

def half_actions(bas_path, half):
    want = "1" if half == "1st" else "2"
    return [a for a in json.load(open(bas_path))["actions"] if str(a["gameTime"]).split(" - ")[0].strip() == want]

OUTD = ROOT + "/data/external/soccertrack"
for m in MATCHES:
    for half in ("1st", "2nd"):
        out = f"{OUTD}/{m}_{half}.npz"
        if os.path.exists(out): continue
        bas = hf_hub_download(REPO, f"bas/{m}/{m}_12_class_events.json", repo_type="dataset", local_dir=TEMP + "/st2", token=TOKEN)
        gp = hf_hub_download(REPO, f"gsr/{m}/{m}_{half}.json", repo_type="dataset", local_dir=TEMP + "/st2", token=TOKEN)
        players, team = read_gsr(gp); n = len(players)
        lab = build_labels(half_actions(bas, half), n)
        np.savez_compressed(out, ball=np.full((n, 2), np.nan, np.float32), players=players, team=team, label=lab,
                            period=np.full(n, 1 if half == "1st" else 2, np.int8))
        os.remove(gp)
        print(m, half, "frames", n, "labelled %.0f%%" % (100 * np.mean(lab >= 0)), flush=True)
print(len(glob.glob(OUTD + "/*.npz")), "SoccerTrack halves ready (need 20)")

## 4. Final training: all data, 5 architectures x 3 seeds
Features are built once, then every model is trained and saved immediately to `/kaggle/working/out/possession_gru_<arch>_final_s<seed>.pt`.
There is no validation here; epoch counts come from the earlier validation runs (edit them at the top if you know better).

In [ ]:
def run(args):
    p = subprocess.Popen([sys.executable, "-u", ROOT + "/training/train_v4.py"] + args, cwd=ROOT,
                         stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    for line in p.stdout:
        print(line, end="", flush=True)
    return p.wait()
archs = ",".join("{}:{}".format(a, e) for a, e in EPOCHS.items())
run(["--final", "--n-in", "35", "--archs", archs, "--seeds", str(SEEDS), "--out-dir", OUTW])
print(sorted(os.listdir(OUTW)))

## 5. Independent test: Metrica public sample games 1 and 2
Labels come from event data (a source none of the models saw). Two conditions per game: ball as tracked, and with an extra simulated outage. Official spell F1 + frame macro F1, per half.
Compare single architectures and ensembles; read the *differences* between rows, not the absolute numbers.

In [ ]:
subprocess.run([sys.executable, ROOT + "/training/build_metrica.py"], check=True)

In [ ]:
def spec(name, archs):
    parts = ["{}={}/possession_gru_{}_final_s*.pt".format(a, OUTW, a) for a in archs if glob.glob("{}/possession_gru_{}_final_s*.pt".format(OUTW, a))]
    return "{}:35:{}".format(name, ",".join(parts)) if parts else None
specs = [spec(a, [a]) for a in ("gru", "lstm", "tcn", "convgru", "transformer")]
specs += [spec("ens_no_lstm", ["gru", "tcn", "convgru", "transformer"]), spec("ens_all", ["gru", "lstm", "tcn", "convgru", "transformer"]),
          spec("ens_tcn_conv", ["tcn", "convgru"])]
specs = [s for s in specs if s]
run_cmd = [sys.executable, "-u", ROOT + "/training/eval_metrica.py", "--seeds", "2"] + specs
p = subprocess.Popen(run_cmd, cwd=ROOT, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
for line in p.stdout: print(line, end="", flush=True)
p.wait()